In [1]:
# For statistical analysis
import pandas as pd, os, datetime
import numpy as np
from scipy import stats
from scipy.stats import mannwhitneyu

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
def jitter(group):
    duplicates = group.groupby(['lat', 'lon']).cumcount()
    
    # Jitter function: add a small random offset
    np.random.seed(42)  # For reproducibility
    jitter_strength = 0.05 # Adjust as needed; degrees latitude/longitude
    
    group['lat_jittered'] = group['lat'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates
    group['lon_jittered'] = group['lon'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(group)) * duplicates

    return group

In [4]:
gen_details = pd.read_csv(f"{nmap_path}/gen_info.csv")

In [5]:
# A temporary patch to fix fuel types
gen_details['fuel_source_primary'] = gen_details['fuel_source_primary'].replace({'Solar - Solar': 'Solar','Wind - Wind': 'Wind'})
gen_details = jitter(gen_details)

In [6]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")

In [7]:
def process_group_hourly(grp, gen_fpath, hw_tseries, start_date=None, end_date=None):
    safe_duids = [duid.replace("/", "_").replace("\\", "_") for duid in grp['DUID']]
    gen_locs = [f"{gen_fpath}/{duid}.csv" for duid in safe_duids]
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]
    print(f"Loaded {len(dfs)} CSV file(s) out of {len(gen_locs)} expected.")

    if not dfs:
        return None

    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]
    
    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[start_date:end_date]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        dfs.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("12h"),  # 1 hour tolerance now
        direction='nearest'
    )

    merged = merged.dropna(how='all')

    return merged.reset_index()


In [8]:
def select_group(gen_details, state=None, ftype=None):
    if state is not None and ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[(gen_details['region'] == state) & (gen_details['fuel_source_primary'].isin(ftype))]
        else:
            groups = gen_details.groupby(['region', 'fuel_source_primary'])
            grp = groups.get_group((state, ftype))
    elif state is not None:
        groups = gen_details.groupby('region')
        grp = groups.get_group(state)
    elif ftype is not None:
        if isinstance(ftype, list):
            grp = gen_details[gen_details['fuel_source_primary'].isin(ftype)]
        else:
            groups = gen_details.groupby('fuel_source_primary')
            grp = groups.get_group(ftype)
    else:
        grp = gen_details

    return grp


In [9]:
sdate, edate = '2009-07-01','2024-06-30'

We remove the hours 2100-0500 from the solar data to prevent left skew

In [10]:
def clean_df(df, gen_details): 
    df = df.merge(gen_details[['DUID', 'reg_cap_mw','technology_type_primary','fuel_source_primary','lat_jittered','lon_jittered', 'region']], on='DUID', how='inner')
    
    djf = [12, 1, 2]  # December, January, February
    df = df[df['time'].dt.month.isin(djf)]
    
    # For solar only: remove rows between 21:00 and 05:00
    solar_mask = df['fuel_source_primary'] == 'Solar'
    times = df.loc[solar_mask, 'time'].dt.time
    
    time_filter = ~(
        (times >= datetime.time(21, 0)) |  # 21:00 onwards
        (times < datetime.time(5, 0))      # before 05:00
    )
    
    # Keep all non-solar rows, and solar rows passing the time filter
    df = pd.concat([
        df[~solar_mask],
        df.loc[solar_mask].loc[time_filter]
    ])
    
    return df

In [11]:
def min_heatwave_days(df, min_days=20):
    """
    Filters the DataFrame to include only DUIDs with at least `min_days`
    of unique heatwave days (EHF_flag == 1).
    
    Parameters:
        df (pd.DataFrame): Input DataFrame with columns ['DUID', 'time', 'EHF_flag']
        min_days (int): Minimum number of unique heatwave days required

    Returns:
        pd.DataFrame: Filtered DataFrame
    """
    df = df.copy()
    df['time'] = pd.to_datetime(df['time'])
    df['date'] = df['time'].dt.date

    # Count unique heatwave days per DUID
    heatwave_days = (
        df[df['EHF_flag'] == 1]
        .groupby('DUID')['date']
        .nunique()
    )

    # Keep only DUIDs meeting the threshold
    valid_duids = heatwave_days[heatwave_days >= min_days].index
    
    return df[df['DUID'].isin(valid_duids)].copy()


In [12]:
def remove_wind_zero_rows(df, duid_col='DUID', value_col='TOTALMWh',
                          tech_col='fuel_source_primary', wind_label='Wind', threshold=5):
    """
    Remove *all rows* for wind DUIDs where the percentage of zeros 
    exceeds the given threshold. Non-wind DUIDs are left untouched.
    
    Parameters:
        df (pd.DataFrame): Input dataframe
        duid_col (str): Column name for grouping (default 'DUID')
        value_col (str): Column name with numeric values (default 'TOTALMWh')
        tech_col (str): Column that identifies technology type (default 'fuel_source_primary')
        wind_label (str): Label used for wind in tech_col (default 'Wind')
        threshold (float): Maximum allowed percentage of zeros (default 5)
    
    Returns:
        pd.DataFrame: Filtered dataframe
    """
    
    # work only on wind rows
    wind_df = df[df[tech_col] == wind_label]
    
    # calculate % of zeros per wind DUID
    percent_zeros = (
        wind_df.groupby(duid_col)[value_col]
               .apply(lambda x: (x == 0).sum() / len(x) * 100)
    )
    
    # keep DUIDs below threshold
    keep_duids = percent_zeros[percent_zeros <= threshold].index
    
    # filter wind df
    wind_filtered = wind_df[wind_df[duid_col].isin(keep_duids)]
    
    # keep all non-wind rows
    non_wind_df = df[df[tech_col] != wind_label]
    
    # combine and return
    return pd.concat([non_wind_df, wind_filtered], ignore_index=True)

In [ ]:
info = select_group(gen_details,ftype=['Wind','Solar']).copy()
df = process_group_hourly(info, gen_fpath, hw_tseries,sdate,edate)
df = clean_df(df,info)
df = remove_wind_zero_rows(df, threshold=40)
df = min_heatwave_days(df,20)

df['norm_std'] = df.groupby('DUID')['TOTALMWh'].transform(lambda x: (x - x.median()) / x.std(ddof=0))

In [ ]:
df = df.copy()
df['time'] = pd.to_datetime(df['time'])
df['date'] = df['time'].dt.date

# Count unique heatwave days per DUID
heatwave_days = (
    df[df['EHF_flag'] == 1]
    .groupby('DUID')['date']
    .nunique()
)
heatwave_days

In [3]:
def hourly_mannwhitu(df, var):
    df = df.copy()
    results = []
    df['hour'] = df['time'].dt.hour
    
    for duid, group in df.groupby('DUID'):
        for hour in range(24):
            group_hour = group[group['hour'] == hour]
            
            group_0 = group_hour[group_hour['EHF_flag'] == 0][var].dropna()
            group_1 = group_hour[group_hour['EHF_flag'] == 1][var].dropna()
            
            # print(f"DUID: {duid}, hour: {hour}, size 0: {len(group_0)}, size 1: {len(group_1)}")
            
            if len(group_0) < 10 or len(group_1) < 10:
                continue
            
            t_stat, p_val = mannwhitneyu(group_0, group_1, alternative='two-sided')
            
            results.append({
                'DUID': duid,
                'hour': hour,
                't_statistic': t_stat,
                'p_value': p_val,
                'median_baseline': group_0.median(),
                'median_hw': group_1.median()
            })
    
    result_df = pd.DataFrame(results)
    print("Unique hours in results:", np.unique(result_df['hour']))
    return result_df


In [4]:
def plot_hourly_mannwhitu_map(df, info, mannwhitu_results, alpha=0.05,ftypes='Wind and Solar'):
    merged = mannwhitu_results.merge(info, on='DUID', how='left').dropna(subset=['lat_jittered', 'lon_jittered'])
    merged['change'] = merged['median_hw'] - merged['median_baseline']
    merged['size'] = (merged['change'].abs().replace(0, 0.001) * 100) + 20

    def pvalue_significance(p):
        if p < alpha / 10:
            return 'Very strong'
        elif p < alpha:
            return 'Strong'
        elif p < 0.1:
            return 'Weak'
        else:
            return 'Not significant'

    merged['SignificanceLevel'] = merged['p_value'].apply(pvalue_significance)

    def category(row):
        sig = row['SignificanceLevel']
        if sig == 'Not significant':
            return sig
        elif row['change'] < 0:
            return f'{sig} decrease'
        else:
            return f'{sig} increase'

    merged['Category'] = merged.apply(category, axis=1)

    color_map = {
        'Very strong increase': 'darkgreen',
        'Strong increase': 'green',
        'Weak increase': 'lightgreen',
        'Very strong decrease': 'darkred',
        'Strong decrease': 'red',
        'Weak decrease': 'salmon',
        'Not significant': 'lightgray'
    }

    # Ensure 'hour' is string for animation frame to work nicely
    merged['hour'] = merged['hour'].astype(str)

    fig = px.scatter_map(
        merged,
        lat='lat_jittered',
        lon='lon_jittered',
        size='size',
        color='Category',
        color_discrete_map=color_map,
        hover_name='DUID',
        hover_data={
            'hour': True,
            'change': ':.2f',
            'median_baseline': ':.2f',
            'median_hw': ':.2f',
            'p_value': ':.4f',
            'lat_jittered': False,
            'lon_jittered': False,
            'size': False
        },
        zoom=5,
        map_style='carto-darkmatter',
        animation_frame='hour',
        title=f'Hourly Results: Shift in generation distribution for {ftypes}'
    )

    fig.update_layout(
        legend_title_text='Significance and Direction',
        margin=dict(l=10, r=10, t=50, b=10),
        map=dict(center=dict(lat=merged['lat_jittered'].median(), lon=merged['lon_jittered'].median()))
    )

    fig.show()
    return fig


In [5]:
mannwhitu_hourly_results = hourly_mannwhitu(df[df['fuel_source_primary'] == 'Wind'],'TOTALMWh')
fig = plot_hourly_mannwhitu_map(df[df['fuel_source_primary'] == 'Wind'], info, mannwhitu_hourly_results,ftypes='Wind')
fig.write_html("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hourly_mannwhitu_wind.html")

NameError: name 'df' is not defined

In [6]:
solar_hourly_results = hourly_mannwhitu(df[df['fuel_source_primary'] == 'Solar'], 'TOTALMWh')
fig = plot_hourly_mannwhitu_map(df[df['fuel_source_primary'] == 'Solar'], info, solar_hourly_results,ftypes='Solar')
fig.write_html("/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/hourly_mannwhitu_solar.html")

NameError: name 'df' is not defined

In [ ]:
def iqr_ribbon_plot_totalmwh_dropdown(df):
    df = df.copy()
    df['Hour'] = df['time'].dt.hour
    df['TimeLabel'] = df['Hour'].apply(lambda h: f"{h:02d}:00")

    duids = sorted(df['DUID'].unique())
    fig = go.Figure()
    trace_index = 0
    duid_trace_map = {}  # Keeps track of trace indices for each DUID

    for duid in duids:
        duid_df = df[df['DUID'] == duid]
        duid_traces = []

        for flag_value in sorted(duid_df['EHF_flag'].dropna().unique()):
            group = duid_df[duid_df['EHF_flag'] == flag_value]
            summary = group.groupby(['Hour', 'TimeLabel'])['TOTALMWh'].agg(
                Q1=lambda x: x.quantile(0.25),
                Q3=lambda x: x.quantile(0.75),
                median='median'
            ).reset_index().sort_values('Hour')

            # Q3 (invisible)
            fig.add_trace(go.Scatter(
                x=summary['TimeLabel'], y=summary['Q3'],
                line=dict(width=0),
                showlegend=False,
                visible=(duid == duids[0]),  # Only show first DUID initially
                mode='lines'
            ))
            duid_traces.append(trace_index)
            trace_index += 1

            # Q1 fill
            fig.add_trace(go.Scatter(
                x=summary['TimeLabel'], y=summary['Q1'],
                fill='tonexty',
                fillcolor=f'rgba({50 + 100 * flag_value}, 100, 255, 0.2)',
                line=dict(width=0),
                name=f'IQR (Q1–Q3), EHF_flag = {flag_value}',
                visible=(duid == duids[0]),
                mode='lines'
            ))
            duid_traces.append(trace_index)
            trace_index += 1

            # median line
            fig.add_trace(go.Scatter(
                x=summary['TimeLabel'], y=summary['median'],
                line=dict(width=2),
                name=f'median, EHF_flag = {flag_value}',
                visible=(duid == duids[0]),
                mode='lines'
            ))
            duid_traces.append(trace_index)
            trace_index += 1

        duid_trace_map[duid] = duid_traces

    # Create dropdown buttons
    buttons = []
    total_traces = trace_index

    for duid in duids:
        visibility = [False] * total_traces
        for idx in duid_trace_map[duid]:
            visibility[idx] = True

        buttons.append(dict(
            label=duid,
            method='update',
            args=[
                {'visible': visibility},
                {'title.text': f'IQR + median TOTALMWh by Hour for DUID: {duid}'}
                
            ]
        ))

    fig.update_layout(
        updatemenus=[dict(
            buttons=buttons,
            direction='down',
            showactive=True,
            x=1.1,
            xanchor='center',
            y=1.15,
            yanchor='top'
        )],
        title=f'IQR + median TOTALMWh by Hour for DUID: {duids[0]}',
        xaxis_title='Hour of Day',
        yaxis_title='TOTALMWh',
        xaxis_tickangle=-45,
        template='plotly_white'
    )

    fig.show()

iqr_ribbon_plot_totalmwh_dropdown(df[df['fuel_source_primary'] == 'Solar'])

In [ ]:
# def boxplot_buckets(df, value_col='TOTALMWh', event_day_col='HW_event_day',
#                     category_col=None, category_value=None):
#     df_plot = df.copy()

#     # Optional category filtering
#     if category_col and category_value:
#         df_plot = df_plot[df_plot[category_col] == category_value]

#     # Create buckets: 1,2,3,last
#     df_plot['bucket'] = df_plot[event_day_col].apply(
#         lambda x: 'NaN' if pd.isna(x) else ('last' if x > 3 else str(int(x)))
#     )

#     # Keep only relevant buckets
#     df_plot = df_plot[df_plot['bucket'].isin(['1','2','3','last'])]

#     # Create box plot
#     fig = px.violin(
#         df_plot,
#         x='bucket',
#         y=value_col,
#         points='all',
#         color='bucket',
#         hover_data=['DUID'],
#         title=f"{value_col} distribution per event bucket" + (f" for {category_value}" if category_value else "")
#     )
#     fig.show()

# boxplot_buckets(df, category_col='fuel_source_primary', value_col='norm_std', category_value='Wind')